# 🎓 Day 1 · Session 5
# 05A · RAG Fundamentals
## Why RAG exists, how embeddings retrieve meaning, and how grounding works

## 🎯 Learning Objectives

- Explain why RAG is needed
- Understand grounding, embeddings, chunking and vector search
- Build a mini semantic search example

## 🔧 Setup

Expected `.env` file in the same folder as this notebook:

```env
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas numpy scikit-learn pypdf chromadb langchain langchain-openai langchain-community langchain-chroma

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import pandas as pd
import numpy as np
from openai import OpenAI

In [ ]:
env_path = Path.cwd() / ".env"

print("Current working directory:", Path.cwd())
print("Looking for .env at:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Dotenv keys found:", list(dotenv_values(env_path).keys()))
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized successfully.")
print("Key starts with:", api_key[:10], "...")

In [ ]:
def ask_openai(prompt, model="gpt-4o-mini", temperature=0.2, system_prompt="You are a helpful academic assistant.", max_tokens=700):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# 1️⃣ Why RAG?

LLMs are powerful, but they do not automatically know your private documents.

They may not know:
- your college syllabus
- your fee structure
- your department handbook
- your lab manual
- your internal policies
- your latest circulars

RAG solves this by giving the LLM the right documents at query time.

> RAG = Retrieval-Augmented Generation  
> LLM + Retrieved Context = Grounded Answer

In [ ]:
question = "What is the M.Tech CSE fee at ABC Institute of Technology?"

print(ask_openai(question))

## 🔍 Observe

The answer is likely generic.

The model does not know your institution's actual fee structure unless you provide the document.

This is the exact problem RAG solves.

# 2️⃣ Grounding

Grounding means forcing the model to answer from a trusted source.

Without grounding:

```text
Question → LLM → Generic / guessed answer
```

With grounding:

```text
Question → Retrieve relevant document chunks → LLM reads chunks → Grounded answer
```

In [ ]:
college_handbook = '''
ABC Institute of Technology - Department Handbook

M.Tech CSE Fee Structure:
The tuition fee for M.Tech CSE is Rs. 50,000 per semester.
The programme duration is 4 semesters.
GATE-qualified students are eligible for a 50% tuition scholarship.

NLP Elective:
The NLP elective is offered in Semester 3.
Prerequisites: Machine Learning and Python Programming.
Faculty Coordinator: Dr. Kavitha Raman.

Attendance Rule:
Students must maintain 75% attendance to be eligible for final examinations.
'''

rag_prompt = f'''
Answer the question using ONLY the context below.
If the answer is not present in the context, say:
"I don't have this information in the provided knowledge base."

Context:
{college_handbook}

Question:
What is the M.Tech CSE fee?
'''

print(ask_openai(rag_prompt))

# 3️⃣ Semantic Search vs Keyword Search

Keyword search looks for exact words.

Semantic search looks for meaning.

Example:
- Query: "cost of masters programme"
- Document: "M.Tech CSE tuition fee is Rs. 50,000 per semester"

A keyword search may fail.  
Semantic search can still find the relevant chunk.

In [ ]:
documents = [
    "M.Tech CSE tuition fee is Rs. 50,000 per semester.",
    "The NLP elective requires Machine Learning and Python Programming.",
    "Students must maintain 75% attendance.",
    "The campus library is open from 8 AM to 8 PM.",
    "The AI lab has GPU workstations for deep learning experiments."
]

query = "cost of masters programme"

print("Query:", query)
print("\nDocuments:")
for i, doc in enumerate(documents, 1):
    print(i, doc)

# 4️⃣ Embeddings

Embeddings convert text into vectors.

Similar meaning → similar vectors → closer in vector space.

In RAG:
1. Convert document chunks into embeddings
2. Store embeddings in vector database
3. Convert user question into embedding
4. Retrieve closest chunks

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

query_embedding = get_embedding(query)
doc_embeddings = [get_embedding(doc) for doc in documents]

print("Embedding dimension:", len(query_embedding))
print("Number of document embeddings:", len(doc_embeddings))

In [ ]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = []
for doc, emb in zip(documents, doc_embeddings):
    scores.append({
        "document": doc,
        "similarity": cosine_similarity(query_embedding, emb)
    })

pd.DataFrame(scores).sort_values("similarity", ascending=False)

# 5️⃣ Chunking

Large documents are split into smaller pieces called chunks.

Why?

- LLM context is limited
- Retrieval works better on smaller units
- We only send relevant parts to the model

Common starting point:
- chunk size: 400–800 characters
- overlap: 50–150 characters

In [ ]:
sample_text = college_handbook

def simple_chunk_text(text, chunk_size=250, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = simple_chunk_text(sample_text, chunk_size=250, overlap=50)

for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

# 6️⃣ RAG Architecture

```text
Documents
   ↓
Chunking
   ↓
Embeddings
   ↓
Vector Store
   ↓
Retriever
   ↓
Prompt with Context
   ↓
LLM
   ↓
Grounded Answer
```

# 🧪 Mini Lab

Try these questions:

1. What are the prerequisites for NLP?
2. Who coordinates NLP?
3. What is the attendance requirement?
4. What is the hostel fee?

Expected: The first three should answer from context.  
The fourth should say information is not available.

In [ ]:
def answer_from_context(question, context):
    prompt = f'''
You are a helpful academic assistant.
Answer using ONLY the context below.

If the answer is not in the context, say:
"I don't have this information in the provided knowledge base."

Context:
{context}

Question:
{question}
'''
    return ask_openai(prompt)

questions = [
    "What are the prerequisites for NLP?",
    "Who coordinates NLP?",
    "What is the attendance requirement?",
    "What is the hostel fee?"
]

for q in questions:
    print("\nQUESTION:", q)
    print("ANSWER:", answer_from_context(q, college_handbook))

# ✅ Summary

You learnt:

- RAG gives the LLM access to your trusted documents
- Embeddings search by meaning
- Chunking makes documents retrievable
- The RAG prompt must force the model to answer only from context
- RAG is the foundation for document-aware assistants